In [1]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Callable, Optional
from scipy.linalg import expm

# ------------------------------------------------------------
# SeeMPS imports
# ------------------------------------------------------------
from seemps.state import CanonicalMPS, product_state, DEFAULT_STRATEGY
from seemps.expectation import expectation1


# ============================================================
# Basic local operators
# Basis conventions:
#   qubit: |g> = [1,0], |e> = [0,1]
#   bin  : |0> = [1,0], |1> = [0,1]
# ============================================================
def qubit_ops():
    I = np.eye(2, dtype=complex)
    sm = np.array([[0, 1], [0, 0]], dtype=complex)  # |g><e|
    sp = sm.conj().T
    sx = sm + sp
    sz = np.array([[-1, 0], [0, 1]], dtype=complex)
    pe = sp @ sm  # |e><e|
    return I, sm, sp, sx, sz, pe


def bin_ops():
    I = np.eye(2, dtype=complex)
    b = np.array([[0, 1], [0, 0]], dtype=complex)  # hard-core / 0-1 bin truncation
    bd = b.conj().T
    n = bd @ b
    return I, b, bd, n


def swap_2site(dim: int = 2) -> np.ndarray:
    """SWAP gate on two equal-dimension sites."""
    S = np.zeros((dim * dim, dim * dim), dtype=complex)
    for i in range(dim):
        for j in range(dim):
            ket_in = i * dim + j
            ket_out = j * dim + i
            S[ket_out, ket_in] = 1.0
    return S


# ============================================================
# Config
# ============================================================
@dataclass
class DrivenMarkovQubitConfig:
    gamma: float = 1.0
    delta: float = 0.0
    dt: float = 0.02
    tmax: float = 10.0

    # Classical drive:
    # H_drive = (Omega(t)/2) sigma_x
    Omega0: float = 1.0
    drive_func: Optional[Callable[[float], float]] = None

    # MPS truncation
    chi_max: Optional[int] = None
    tol: Optional[float] = None

    # initial qubit state: "g" or "e"
    qubit_init: str = "g"

    def omega(self, t: float) -> float:
        if self.drive_func is None:
            return self.Omega0
        return float(self.drive_func(t))

    @property
    def nsteps(self) -> int:
        return int(round(self.tmax / self.dt))


# ============================================================
# Strategy helper
# ============================================================
def build_strategy(cfg: DrivenMarkovQubitConfig):
    """
    Build a SeeMPS truncation strategy.
    Depending on the exact seemps version, the replace(...) keywords may vary.
    The try/except keeps this script robust across minor API changes.
    """
    strategy = DEFAULT_STRATEGY

    # We do not want automatic renormalization after every split here.
    try:
        strategy = strategy.replace(normalize=False)
    except Exception:
        pass

    if cfg.chi_max is not None:
        for key in ("max_bond_dimension", "max_bond", "chi_max"):
            try:
                strategy = strategy.replace(**{key: int(cfg.chi_max)})
                break
            except Exception:
                pass

    if cfg.tol is not None:
        for key in ("tolerance", "tol"):
            try:
                strategy = strategy.replace(**{key: float(cfg.tol)})
                break
            except Exception:
                pass

    return strategy


# ============================================================
# Initial MPS
# Chain layout:
#   site 0      : system qubit
#   site 1..N   : time bins b_0, b_1, ..., b_{N-1}
#
# Each bin starts in vacuum |0>.
# ============================================================
def build_initial_mps(cfg: DrivenMarkovQubitConfig, strategy=None) -> CanonicalMPS:
    if strategy is None:
        strategy = build_strategy(cfg)

    g = np.array([1.0, 0.0], dtype=complex)
    e = np.array([0.0, 1.0], dtype=complex)
    vac = np.array([1.0, 0.0], dtype=complex)

    q0 = g if cfg.qubit_init.lower() == "g" else e
    local_states = [q0] + [vac for _ in range(cfg.nsteps)]

    psi = product_state(local_states)
    psi = CanonicalMPS(psi, center=0, strategy=strategy)
    return psi


# ============================================================
# Local pair Hamiltonian and gate
#
# Pair ordering before gate:
#   left  site = current system
#   right site = fresh incoming time bin
#
# H_pair = H_sys(t) \otimes I_bin
#          + i sqrt(gamma/dt) ( sigma_- \otimes b^\dagger - sigma_+ \otimes b )
#
# Then we apply SWAP after the collision:
#   G = SWAP @ exp(-i H_pair dt)
#
# so that:
#   left output  = emitted/output bin
#   right output = updated system
#
# This lets the system "walk" to the right through the chain.
# ============================================================
def build_pair_gate(t_mid: float, cfg: DrivenMarkovQubitConfig) -> np.ndarray:
    Iq, sm, sp, sx, sz, _ = qubit_ops()
    Ib, b, bd, _ = bin_ops()

    H_sys = 0.5 * cfg.delta * sz + 0.5 * cfg.omega(t_mid) * sx
    H_int = 1j * np.sqrt(cfg.gamma / cfg.dt) * (np.kron(sm, bd) - np.kron(sp, b))

    H_pair = np.kron(H_sys, Ib) + H_int
    U = expm(-1j * H_pair * cfg.dt)

    S = swap_2site(2)
    G = S @ U

    # reshape as G[out_left, out_right, in_left, in_right]
    return G.reshape(2, 2, 2, 2)


# ============================================================
# Apply a nearest-neighbor two-site gate and move canonical center right
# ============================================================
def apply_two_site_gate_right(
    psi: CanonicalMPS,
    site: int,
    gate4: np.ndarray,
    strategy,
) -> CanonicalMPS:
    """
    gate4 has indices [out_i, out_j, in_i, in_j].
    After application, use update_2site_right(...) so the center moves to site+1.
    """
    if psi.center != site:
        psi.recenter(site, strategy=strategy)

    A = psi[site]  # [a, i, b]
    B = psi[site + 1]  # [b, j, c]

    # form two-site tensor Theta[a, i, j, c]
    Theta = np.einsum("aib,bjc->aijc", A, B, optimize=True)

    # apply gate on physical legs
    Theta_new = np.einsum("xyij,aijc->axyc", gate4, Theta, optimize=True)

    # split back; canonical center becomes site+1
    psi.update_2site_right(Theta_new, site, strategy)
    return psi


# ============================================================
# Main evolution
# ============================================================
def t_evol_mar_driven_qubit_seemps(cfg: DrivenMarkovQubitConfig):
    """
    Markovian evolution of a driven qubit coupled to a single waveguide channel.

    Returns
    -------
    result : dict
        keys:
          - t_sys          : times for qubit population, length N+1
          - p_exc          : excited-state population, length N+1
          - t_out          : times for emitted bins, length N
          - n_out          : photon number in each output bin, length N
          - flux_out       : n_out / dt
          - final_state    : final CanonicalMPS
    """
    strategy = build_strategy(cfg)
    psi = build_initial_mps(cfg, strategy=strategy)

    _, _, _, _, _, pe = qubit_ops()
    _, _, _, nbin = bin_ops()

    N = cfg.nsteps
    dt = cfg.dt

    t_sys = np.arange(N + 1) * dt
    t_out = (np.arange(N) + 1) * dt

    p_exc = np.zeros(N + 1, dtype=float)
    n_out = np.zeros(N, dtype=float)

    # initially the system is at site 0
    p_exc[0] = np.real(expectation1(psi, pe, 0))

    # At step k:
    #   current system site = k
    #   fresh bin site      = k+1
    for k in range(N):
        t_mid = (k + 0.5) * dt
        gate4 = build_pair_gate(t_mid, cfg)

        psi = apply_two_site_gate_right(psi, k, gate4, strategy)

        # after SWAP@U:
        #   site k     is now the output bin
        #   site k+1   is the updated system
        n_out[k] = np.real(expectation1(psi, nbin, k))
        p_exc[k + 1] = np.real(expectation1(psi, pe, k + 1))

    flux_out = n_out / dt

    return {
        "t_sys": t_sys,
        "p_exc": p_exc,
        "t_out": t_out,
        "n_out": n_out,
        "flux_out": flux_out,
        "final_state": psi,
    }


# ============================================================
# Example drive functions
# ============================================================
def constant_drive(Omega0: float) -> Callable[[float], float]:
    return lambda t: Omega0


def sinusoidal_drive(
    Omega0: float, wd: float, phase: float = 0.0
) -> Callable[[float], float]:
    return lambda t: Omega0 * np.cos(wd * t + phase)


def square_drive(Omega0: float, period: float) -> Callable[[float], float]:
    return lambda t: Omega0 if (t % period) < 0.5 * period else -Omega0


# ============================================================
# Example usage
# ============================================================
if __name__ == "__main__":
    cfg = DrivenMarkovQubitConfig(
        gamma=1.0,
        delta=0.0,
        dt=0.02,
        tmax=8.0,
        drive_func=sinusoidal_drive(Omega0=1.2, wd=1.0),
        chi_max=64,
        tol=1e-10,
        qubit_init="g",
    )

    res = t_evol_mar_driven_qubit_seemps(cfg)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(res["t_sys"], res["p_exc"], lw=2, label=r"$P_e(t)$")
    ax.plot(res["t_out"], res["flux_out"], lw=2, label=r"$\Phi_{\rm out}(t)$")
    ax.set_xlabel("t")
    ax.set_ylabel("population / flux")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("Final excited population =", res["p_exc"][-1])
    print("Total emitted quanta     =", np.sum(res["n_out"]))

ModuleNotFoundError: No module named 'seemps'